# Analisis de precios a partir de SEPA: obtencion de datos
Basado en base [SEPA](https://datos.produccion.gob.ar/dataset/sepa-precios)

In [1]:
# =============================================================================
# ANÁLISIS SEPA — VERSIÓN OPTIMIZADA Y CONSOLIDADA
# =============================================================================
# Reorganización del notebook original para minimizar uso de RAM y permitir
# recuperación rápida tras crashes. Cambios clave vs versión previa:
#
#   1. PERSISTENCIA A DISCO: cada bloque grande guarda su output a parquet,
#      así si el kernel muere no hay que re-descomprimir ni re-cargar.
#   2. LIBERACIÓN EXPLÍCITA: del + gc.collect() después de cada etapa.
#   3. ELIMINACIÓN DE DUPLICIDADES: df_productos se descarta al construir
#      df_precios (son los mismos datos + joins).
#   4. CLASIFICADOR CONSOLIDADO: una sola función final (v4 + fixes) en vez
#      de aplicar 4 versiones sucesivas.
#   5. DIAGNÓSTICOS OPCIONALES: los prints que no son críticos están en
#      bloques con flag; se pueden saltear para correr rápido.
# =============================================================================

import os
import re
import gc
import zipfile
import unicodedata
from io import StringIO
from pathlib import Path

import pandas as pd
import numpy as np


# =============================================================================
# CONFIGURACIÓN Y CONSTANTES
# =============================================================================

CONTENT_DIR = Path("/content")
WORK_DIR    = Path("/content/sepa_data")
CACHE_DIR   = Path("/content/sepa_cache")  # Parquets para recuperar sin re-cargar
WORK_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

# Flag para saltear diagnósticos pesados (ponelo en False si querés correr rápido)
DIAGNOSTICOS = True

# Mapa provincia ISO-3166 → nombre
ISO_A_PROVINCIA = {
    "AR-C": "CABA", "AR-B": "Buenos Aires", "AR-K": "Catamarca",
    "AR-H": "Chaco", "AR-U": "Chubut", "AR-X": "Córdoba",
    "AR-W": "Corrientes", "AR-E": "Entre Ríos", "AR-P": "Formosa",
    "AR-Y": "Jujuy", "AR-L": "La Pampa", "AR-F": "La Rioja",
    "AR-M": "Mendoza", "AR-N": "Misiones", "AR-Q": "Neuquén",
    "AR-R": "Río Negro", "AR-A": "Salta", "AR-J": "San Juan",
    "AR-D": "San Luis", "AR-Z": "Santa Cruz", "AR-S": "Santa Fe",
    "AR-G": "Santiago del Estero", "AR-V": "Tierra del Fuego",
    "AR-T": "Tucumán",
}

# Banderas a excluir (farmacias + estaciones de servicio)
BANDERAS_EXCLUIDAS = {
    "FARMACITY", "SIMPLICITY", "FULL", "Axion Energy", "ESTACION LIMA S.A.",
}

# Filtros de calidad para canastables
FILTRO_BANDERAS_MIN   = 8
FILTRO_PROVINCIAS_MIN = 5
FILTRO_RATIO_MAX      = 3.0
FILTRO_PRECIO_MIN     = 50.0


# =============================================================================
# HELPERS GENERALES
# =============================================================================

def reportar_memoria(etiqueta=""):
    """Imprime uso de memoria de todos los DataFrames en globals()."""
    total = 0
    for name, obj in list(globals().items()):
        if isinstance(obj, pd.DataFrame):
            mb = obj.memory_usage(deep=True).sum() / (1024**2)
            total += mb
            if mb > 1:  # solo muestro los que pesan más de 1 MB
                print(f"   {name:<25} {mb:>8.1f} MB  ({len(obj):>12,} filas)")
    print(f"   {'TOTAL':<25} {total:>8.1f} MB  {etiqueta}")


def liberar(*nombres):
    """Borra variables del namespace global y fuerza gc."""
    for n in nombres:
        if n in globals():
            del globals()[n]
    gc.collect()


def normalizar_texto(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return s


def contiene(texto, palabras):
    return any(p in texto for p in palabras)


# =============================================================================
# PASO 1: DESCOMPRESIÓN (idempotente)
# =============================================================================

def paso_1_descomprimir():
    """Descomprime el ZIP del día y los ZIPs internos. Idempotente."""
    patron_fecha = re.compile(r"^(\d{4}-\d{2}-\d{2})\.zip$")
    zips_dia = [p for p in CONTENT_DIR.iterdir()
                if p.is_file() and patron_fecha.match(p.name)]
    if not zips_dia:
        raise FileNotFoundError("No se encontró AAAA-MM-DD.zip en /content/")

    zip_dia   = sorted(zips_dia)[-1]
    fecha_str = patron_fecha.match(zip_dia.name).group(1)
    dia_dir   = WORK_DIR / fecha_str
    dia_dir.mkdir(exist_ok=True)

    print(f"📦 ZIP del día: {zip_dia.name} | Fecha: {fecha_str}")

    # Descompresión externa (solo si no hay ZIPs internos todavía)
    if not list(dia_dir.rglob("*.zip")):
        with zipfile.ZipFile(zip_dia, "r") as z:
            z.extractall(dia_dir)
        print(f"   ✅ Descomprimido externo → {dia_dir}")
    else:
        print(f"   ℹ️  ZIPs internos ya presentes, salteo descompresión externa")

    # Descompresión de los ZIPs internos
    zips_internos = list(dia_dir.rglob("*.zip"))
    for zi in zips_internos:
        destino = dia_dir / zi.stem
        if destino.exists() and any(destino.iterdir()):
            continue
        destino.mkdir(exist_ok=True)
        try:
            with zipfile.ZipFile(zi, "r") as z:
                z.extractall(destino)
        except zipfile.BadZipFile:
            print(f"   ⚠️  ZIP corrupto: {zi.name}")

    # Inventario
    CSVS = {"comercio.csv", "sucursales.csv", "productos.csv"}
    carpetas = [c for c in dia_dir.rglob("*")
                if c.is_dir() and {f.name for f in c.glob("*.csv")} & CSVS]
    print(f"   📂 Carpetas de cadenas: {len(carpetas)}")

    return fecha_str, dia_dir, carpetas


# =============================================================================
# PASO 2: CARGA DE CSVs CON TIPOS OPTIMIZADOS DESDE EL ARRANQUE
# =============================================================================
# Cambio vs versión original: aplicamos los dtypes optimizados (category,
# int32, float32, bool) directamente al cargar, en vez de hacerlo después.
# Esto evita tener el DataFrame "gordo" nunca en memoria.

# Dtypes para lectura inicial (usamos tipos que pandas puede leer directo)
DTYPES_COMERCIO_RAW = {
    "id_comercio": "Int64", "id_bandera": "Int64",
    "comercio_cuit": "string", "comercio_razon_social": "string",
    "comercio_bandera_nombre": "string", "comercio_bandera_url": "string",
    "comercio_version_sepa": "string",
}

DTYPES_SUCURSALES_RAW = {
    "id_comercio": "Int64", "id_bandera": "Int64", "id_sucursal": "Int64",
    "sucursales_nombre": "string", "sucursales_tipo": "string",
    "sucursales_calle": "string", "sucursales_numero": "string",
    "sucursales_latitud": "float64", "sucursales_longitud": "float64",
    "sucursales_observaciones": "string", "sucursales_barrio": "string",
    "sucursales_codigo_postal": "string", "sucursales_localidad": "string",
    "sucursales_provincia": "string",
}

DTYPES_PRODUCTOS_RAW = {
    "id_comercio": "Int64", "id_bandera": "Int64", "id_sucursal": "Int64",
    "id_producto": "string", "productos_ean": "string",
    "productos_descripcion": "string",
    "productos_cantidad_presentacion": "float64",
    "productos_unidad_medida_presentacion": "string",
    "productos_marca": "string",
    "productos_precio_lista": "float64",
    "productos_precio_referencia": "float64",
    "productos_cantidad_referencia": "float64",
    "productos_unidad_medida_referencia": "string",
    # Columnas de promo las cargamos como string; las descartamos al final
    "productos_precio_unitario_promo1": "string",
    "productos_leyenda_promo1": "string",
    "productos_precio_unitario_promo2": "string",
    "productos_leyenda_promo2": "string",
}


def leer_csv_sepa(ruta, dtypes):
    """Lee un CSV de SEPA filtrando líneas no-datos (Última actualización, etc)."""
    try:
        with open(ruta, "r", encoding="utf-8") as f:
            texto = f.read()
    except UnicodeDecodeError:
        with open(ruta, "r", encoding="latin-1") as f:
            texto = f.read()

    lineas = texto.splitlines()
    if not lineas:
        return pd.DataFrame()

    header = lineas[0]
    n_cols = header.count("|") + 1

    limpias = [header]
    for l in lineas[1:]:
        if l.strip() and l.count("|") >= n_cols - 1:
            limpias.append(l)

    return pd.read_csv(
        StringIO("\n".join(limpias)),
        sep="|", dtype=dtypes, on_bad_lines="skip",
        na_values=["", "N/A", "NA", "NaN"],
    )


def paso_2_cargar_y_optimizar(carpetas):
    """
    Carga los tres CSVs de todas las cadenas. Optimiza df_productos
    AGRESIVAMENTE antes de concatenar, así nunca materializa el DF "gordo".
    """
    print("\n⏳ Cargando CSVs de todas las cadenas...")

    lista_com, lista_suc, lista_prod = [], [], []

    for i, carpeta in enumerate(carpetas, 1):
        nombre = carpeta.name

        # comercio.csv
        f = carpeta / "comercio.csv"
        if f.exists():
            df = leer_csv_sepa(f, DTYPES_COMERCIO_RAW)
            df["cadena_folder"] = nombre
            lista_com.append(df)

        # sucursales.csv
        f = carpeta / "sucursales.csv"
        if f.exists():
            df = leer_csv_sepa(f, DTYPES_SUCURSALES_RAW)
            df["cadena_folder"] = nombre
            lista_suc.append(df)

        # productos.csv — acá optimizamos INMEDIATAMENTE
        f = carpeta / "productos.csv"
        if f.exists():
            df = leer_csv_sepa(f, DTYPES_PRODUCTOS_RAW)

            # Optimización temprana: descartamos columnas de promos ya
            df = df.drop(columns=[c for c in df.columns
                                   if "promo" in c.lower() or "leyenda" in c.lower()],
                         errors="ignore")

            # Filtro de calidad: precio > 0 ya desde la carga
            df = df[df["productos_precio_lista"].notna() &
                     (df["productos_precio_lista"] > 0)]

            # IDs → int32 (de Int64 a int32 ahorra mitad de memoria)
            for col in ["id_comercio", "id_bandera", "id_sucursal"]:
                df[col] = df[col].fillna(-1).astype("int32")

            # productos_ean → bool (era "0"/"1")
            df["tiene_ean"] = (df["productos_ean"] == "1")
            df = df.drop(columns=["productos_ean"])

            # Floats → float32
            for col in ["productos_cantidad_presentacion", "productos_precio_lista",
                        "productos_precio_referencia", "productos_cantidad_referencia"]:
                if col in df.columns:
                    df[col] = df[col].astype("float32")

            df["cadena_folder"] = nombre
            lista_prod.append(df)
            print(f"   [{i:2d}/{len(carpetas)}] {nombre[:50]:50s} → {len(df):>9,} filas")

    # Concatenación final
    df_comercio   = pd.concat(lista_com, ignore_index=True) if lista_com else pd.DataFrame()
    df_sucursales = pd.concat(lista_suc, ignore_index=True) if lista_suc else pd.DataFrame()
    df_productos  = pd.concat(lista_prod, ignore_index=True) if lista_prod else pd.DataFrame()

    # Liberar las listas intermedias inmediatamente
    del lista_com, lista_suc, lista_prod
    gc.collect()

    # Categorización post-concat (ahora sabemos las categorías globales)
    cat_cols_prod = ["id_producto", "productos_descripcion", "productos_marca",
                      "productos_unidad_medida_presentacion",
                      "productos_unidad_medida_referencia", "cadena_folder"]
    for col in cat_cols_prod:
        if col in df_productos.columns:
            df_productos[col] = df_productos[col].astype("category")

    print(f"\n✅ Carga completa:")
    print(f"   df_comercio:   {len(df_comercio):,} filas")
    print(f"   df_sucursales: {len(df_sucursales):,} filas")
    print(f"   df_productos:  {len(df_productos):,} filas "
          f"({df_productos.memory_usage(deep=True).sum()/(1024**2):.0f} MB)")

    return df_comercio, df_sucursales, df_productos


# =============================================================================
# PASO 3 (consolidado): FILTROS Y CONSTRUCCIÓN DE df_precios
# =============================================================================
# Cambio clave: descartamos df_productos al crear df_precios. No los tenemos
# ambos en RAM simultáneamente.

def paso_3_construir_df_precios(df_comercio, df_sucursales, df_productos):
    """
    Aplica filtros (farmacias, estaciones) y construye df_precios con JOIN
    a sucursales. Devuelve df_precios + df_sucursales enriquecida.
    df_productos es DESTRUIDO en el proceso (para liberar RAM).
    """
    print("\n🔪 Filtrando banderas no-supermercado...")

    banderas_excl_ids = (df_comercio
        .loc[df_comercio["comercio_bandera_nombre"].isin(BANDERAS_EXCLUIDAS),
             ["id_comercio", "id_bandera"]])
    tuplas_excl = set(zip(banderas_excl_ids["id_comercio"],
                           banderas_excl_ids["id_bandera"]))

    def no_excluida(df):
        m = ~df[["id_comercio", "id_bandera"]].apply(tuple, axis=1).isin(tuplas_excl)
        return df[m].copy()

    df_comercio   = no_excluida(df_comercio)
    df_sucursales = no_excluida(df_sucursales)
    df_productos  = no_excluida(df_productos)

    # Enriquecer sucursales con provincia legible y bandera
    df_sucursales["provincia"] = (df_sucursales["sucursales_provincia"]
                                   .map(ISO_A_PROVINCIA)
                                   .fillna(df_sucursales["sucursales_provincia"]))
    df_sucursales = df_sucursales.merge(
        df_comercio[["id_comercio", "id_bandera",
                     "comercio_razon_social", "comercio_bandera_nombre"]]
            .drop_duplicates(),
        on=["id_comercio", "id_bandera"], how="left",
    )

    # Construir df_precios (merge producto×sucursal)
    print("🧱 Construyendo df_precios...")
    cols_suc = ["id_comercio", "id_bandera", "id_sucursal",
                "comercio_bandera_nombre", "provincia",
                "sucursales_localidad", "sucursales_latitud",
                "sucursales_longitud"]

    df_precios = df_productos.merge(
        df_sucursales[cols_suc],
        on=["id_comercio", "id_bandera", "id_sucursal"],
        how="inner",
    )

    # Convertir nuevas columnas a category
    for col in ["comercio_bandera_nombre", "provincia", "sucursales_localidad"]:
        df_precios[col] = df_precios[col].astype("category")

    # CLAVE: destruir df_productos ahora, ya no lo necesitamos
    del df_productos
    gc.collect()

    mem = df_precios.memory_usage(deep=True).sum() / (1024**2)
    print(f"   df_precios: {len(df_precios):,} filas, {mem:.0f} MB")
    print(f"   (df_productos liberado)")

    return df_comercio, df_sucursales, df_precios


# =============================================================================
# PASO 4: MÉTRICAS POR PRODUCTO Y CANASTABLES
# =============================================================================

def moda_segura(serie):
    m = serie.mode()
    return m.iloc[0] if len(m) > 0 else np.nan


def paso_4_calcular_canastables(df_precios):
    """Calcula productos_metricas y filtra canastables."""
    print("\n🔬 Calculando métricas por producto (1-2 min)...")

    # bandera_id temporal (no lo guardamos como columna del df)
    df_precios["bandera_id"] = (df_precios["id_comercio"].astype(str)
                                 + "_" + df_precios["id_bandera"].astype(str))

    productos_metricas = (df_precios
        .groupby("id_producto", observed=True)
        .agg(
            n_banderas=("bandera_id", "nunique"),
            n_sucursales=("id_sucursal", "nunique"),
            n_provincias=("provincia", "nunique"),
            precio_mediano=("productos_precio_lista", "median"),
            precio_p10=("productos_precio_lista", lambda s: s.quantile(0.10)),
            precio_p90=("productos_precio_lista", lambda s: s.quantile(0.90)),
            descripcion=("productos_descripcion", moda_segura),
            marca=("productos_marca", moda_segura),
            cant_presentacion=("productos_cantidad_presentacion", moda_segura),
            unidad_presentacion=("productos_unidad_medida_presentacion", moda_segura),
        )
        .reset_index())

    productos_metricas["ratio_p90_p10"] = (productos_metricas["precio_p90"]
                                            / productos_metricas["precio_p10"])

    # Limpiamos la columna temporal del df_precios
    df_precios.drop(columns=["bandera_id"], inplace=True)

    # Filtros de calidad
    mask = (
        (productos_metricas["n_banderas"]    >= FILTRO_BANDERAS_MIN) &
        (productos_metricas["n_provincias"]  >= FILTRO_PROVINCIAS_MIN) &
        (productos_metricas["ratio_p90_p10"] <= FILTRO_RATIO_MAX) &
        (productos_metricas["precio_mediano"] >= FILTRO_PRECIO_MIN)
    )
    canastables = productos_metricas[mask].copy()
    canastables["desc_norm"] = canastables["descripcion"].apply(normalizar_texto)

    print(f"   productos_metricas: {len(productos_metricas):,}")
    print(f"   canastables (post-filtros): {len(canastables):,}")

    # Liberar productos_metricas, ya no la necesitamos
    del productos_metricas
    gc.collect()

    return canastables


# =============================================================================
# PASO 5: CLASIFICADOR CONSOLIDADO (v4 + fixes en una sola función)
# =============================================================================

def clasificar_rubro(desc_norm, marca=""):
    """
    Clasificador final consolidado. Sustituye v1, v2, v3, v4 y los fixes
    quirúrgicos del paso 7A.3, todo en una sola pasada.
    """
    t = desc_norm
    m_upper = str(marca).upper() if marca else ""

    # === FIX TEMPRANO por marca (era 7A.3 fix 8) ===
    if m_upper in {"CAT CHOW", "DOG CHOW", "WHISKAS", "PEDIGREE", "EUKANUBA",
                    "PRO PLAN", "EXCELLENT", "GATICAN", "OLD PRINCE",
                    "NUTRIMIX", "TOPNUTRITION"}:
        return "No relevante"
    if "SIEMPRE LIBRE" in m_upper:
        return "Higiene personal"

    # === 1) NO RELEVANTE ===
    if contiene(t, ["cat chow", "dog chow", "alim perro", "alim gato",
                     "aliment perro", "aliment gato", "cachorros",
                     "whiskas", "pedigree", "eukanuba", "pro plan",
                     "perro adult", "gato adult"]):
        return "No relevante"
    if contiene(t, ["crema corp", "crem corp", "crema facial", "crem facial",
                     "crema manos", "crem manos", "crema bronce", "autobronce",
                     "crema rosa", "crem rosa",
                     "mascara pesta", "delineador", "labial", "rimmel",
                     "esmalte", "quitaesm", "maquillaje", "base maquill",
                     "protector labial", "rubor", "sombra de ojos",
                     "colonia ", "perfume ", "fragancia", "eau de",
                     "bodyspray", "body spray"]):
        return "No relevante"
    if contiene(t, ["tic tac", "pastilla sabor", "mentita", "chicle",
                     "beldent", "topline", "halls", "menthoplus"]):
        return "No relevante"
    if contiene(t, ["nafta", "aceite motor", "refrigerante", "limpiaparab",
                     "anticongelante", "lamparita", "pila ", "bateria ",
                     "cargador", "fosforo"]):
        return "No relevante"
    if contiene(t, ["preservativ", "profilact", "test emb",
                     "vela ", "encendedor", "filtro cigarr", "cigarrill"]):
        return "No relevante"

    # === 2) BEBÉ ===
    if contiene(t, ["panal", "pañal", "huggies", "pampers", "babysec"]):
        return "Bebé"
    if contiene(t, ["leche infant", "nutrilon"]) or \
       (contiene(t, ["vital", "laser"]) and contiene(t, ["infant"])):
        return "Bebé"
    if contiene(t, ["johnsons baby", "johnson baby",
                     "toallitas humed bebe", "toallit humed bebe",
                     "shampoo bebe", "shamp bebe", "oleo calcar"]):
        return "Bebé"

    # === 3) ALCOHOL ===
    if contiene(t, ["vino ", "vino\t", "tinto", "malbec", "cabernet",
                     "chardonnay", "sauvignon", "bonarda", "syrah",
                     "cerveza", "quilmes", "brahma", "stella", "heineken",
                     "schneider", "corona", "imperial", "patagonia",
                     "fernet", "branca",
                     "whisky", "whiskey", "johnnie walker", "jhonny walker",
                     "vodka", "smirnoff", " gin ", "beefeater", "bombay",
                     "aperol", "campari", "ron ", "bacardi", "havana",
                     "tequila", "jose cuervo",
                     "aperitivo", "espumante", "champag", "sidra", "sidras",
                     "vermouth", "vermut", "gancia",
                     "amargo ", "amaro", "licor", "cynar", "legui"]):
        return "Alcohol"

    # === 4) HIGIENE PERSONAL ===
    if t.startswith("sh "):  # fix: "SH *" al inicio
        return "Higiene personal"
    if contiene(t, ["shampoo", "shamp ", "aco nutric", "aco reconst",
                     "acondicion", "aco ceramid", "aco balance",
                     "enjuague bucal", "antisept bucal",
                     "pasta dent", "crem dent", "crema dent",
                     "cepillo dent", "cepillo max", "cepillo kids", "cep dent",
                     "hilo dent",
                     "desodor", "desodr", "desod ",
                     "deo aeros", "deo barra", "deo roll",
                     "antitrans", "antitranspir", "roll on", "rollon",
                     "afeit", "gillette", "gilette", "rasurad", "espuma afeit",
                     "protec femen", "protec fem", "toall femen", "toall fem",
                     "toalla feme", "tampon",
                     "protector diar", "protec diar", "diario larg",
                     "jabon toc", "jab toc", "jabon glic", "jabon liq tocad",
                     "jab tocador", "jab glic",
                     "gel intim", "gel int ",
                     "algodon", "hisopo", "cotonete",
                     "disco algod", "discos algod",
                     "crema enjuague", "crem enjuague"]):
        return "Higiene personal"
    # Protec fem (era fix 6)
    if "protec" in t and contiene(t, ["antibact", "leve", "normal", "nocturn",
                                       "alas", "femen"]):
        return "Higiene personal"
    # Pañuelos descartables (era fix 9)
    if ("pañuel" in t or "panuel" in t) and ("descart" in t or "facial" in t):
        return "Higiene personal"

    # === 5) LIMPIEZA ===
    if contiene(t, ["lavandina", "lavand ",
                     "detergente", "deterg ",
                     "lavavajill", "lavav ",
                     "jab polvo", "jab liq", "jabon polvo", "jabon liq",
                     "jab en polvo", "jab p/ropa",
                     "suavizante", "suavizar", "perfumante",
                     "desinfect", "desinf ", "desin fresc", "desin ",
                     "limpiador", "limpiad", "limpi ", "limp liq", "limp ",
                     "limpia ", "limpie", "multiuso",
                     "crem multiuso", "crema multiuso",
                     "papel higi", "pap higi",
                     "rollo cocina", "rollo coc",
                     "servilleta", "servill",
                     "trapo", "trapeador", "esponja", "lampazo",
                     "bolsa consorc", "bolsa residuo", "bolsa resid",
                     "limpia piso", "limpia vidr", "cera para",
                     "destapa", "quitamancha",
                     "pastilla inod", "pastilla p/inod", "past inod",
                     "antihon", "ayudin"]):
        return "Limpieza"

    # === 6) LÁCTEOS ===
    # Fix temprano: "leche *" al inicio (excepto choc/infant)
    if t.startswith("leche ") and not contiene(t, ["choc", "infant",
                                                     "nan 1", "nan 2", "nan 3"]):
        return "Lácteos"
    if contiene(t, ["yogur", "yogh", "yog ", "yog firme", "yog beb",
                     "yogurt", "yog bebi"]):
        return "Lácteos"
    if contiene(t, ["manteca", "mantec ", "margarina", "margar "]):
        return "Lácteos"
    if contiene(t, ["queso", "ques ", "ricota", "mozzarella", "mozarella",
                     "provolone", "muzza", "cremoso", "rallado",
                     "casancrem", "philadelphia"]):
        return "Lácteos"
    if "dulce" in t and "leche" in t:
        return "Lácteos"
    if contiene(t, ["leche ent", "leche desc", "leche parc", "leche fluid",
                     "leche larga", "leche uat", "leche polvo", "leche 3",
                     "leche red", "leche sin lact"]):
        return "Lácteos"
    # Crema de leche y crema para cocinar
    if "crema" in t and contiene(t, ["crema leche", "crema d leche",
                                      "crema d/leche", "crema livia",
                                      "crem livia", "crem p/cocin",
                                      "crema cocinar", "crem cocin",
                                      "crema de lech", "crema tradi",
                                      "crem tradi"]):
        return "Lácteos"
    if "crem" in t and "cocin" in t:  # fix adicional
        return "Lácteos"
    if contiene(t, ["postre ", "flan ready", "flan listo", "danette",
                     "serenito"]):
        return "Lácteos"

    # === 7) CARNES Y FIAMBRES ===
    if contiene(t, [" carne", "carne ", "bife", "asado ", "matambre",
                     "picadillo",
                     "pollo", "pechuga", "pata muslo", "suprema",
                     "cerdo", "bondiola", "panceta",
                     "milanesa", "milan ",
                     "hamb ", "hamburg", "hamburguesa",
                     "salchich", "vienissima", "chorizo", "morcilla",
                     "jamon", " jam ", "jam cocido", "jam crudo",
                     "salame", "salami", "mortadela", "longaniza",
                     "fiambre", " pate ", "pate de", "paté",
                     "atun", " atun", "sardina", "caballa", "anchoa",
                     "salmon ahum", "trucha"]):
        return "Carnes y fiambres"

    # === 8) PANADERÍA Y HARINAS ===
    if contiene(t, ["harina", "harin ",
                     "pan lactal", "pan rallad", "pan rall",
                     "pan de molde", "pan integral", "pan hamb",
                     "tostada", "tostad ", "grissin", "bay biscuit",
                     "gallet", "galletit", "galleta", "galle ", "gall ",
                     "gall.", "gall,",
                     "bizcoch", "factura", "medialuna", "criollito", "talitas",
                     "fideo", "pasta seca", "tallarin", "spaghetti",
                     "mostachol", "tirabuzon", "coditos", "moños",
                     "polenta", "avena", "cereal", "granola",
                     "copos de maiz", "copos maiz",
                     "arroz", "arroz lar", "arroz dob",
                     "levadura",
                     "polvo hornear", "polvo p/horn", "polvo para hornear",
                     "tapa p/tart", "tapa p/empan", "tapa tart", "tapa empan"]):
        return "Panadería y harinas"

    # === 9) ALMACÉN ===
    if t.startswith("sal "):  # fix: "sal *" al inicio
        return "Almacén"
    if contiene(t, ["aceite", "vinagre",
                     "sal fina", "sal grues", "sal entref", " sal ",
                     "azucar", "edulcor",
                     "mayonesa", "ketchup", "mostaza",
                     "salsa", "aderezo", "salsa golf",
                     "tomate ", "tomat ", "tomat.", "pure tom", "pure d tom",
                     "extracto tom", "puré de tom", "pure de tomate",
                     "lenteja", "poroto", "garbanzo", "legumbre", "arveja",
                     "conserv", "mermelad",
                     "caldo", "sopa crem", "sopa ", "gelatina", "gelat",
                     "flan polvo", "flan cajita", "postre polvo",
                     "yerba", "te ", " te\t", "cafe", "cafet", "mate coc",
                     "cocoa", "chocolate en polvo",
                     "oregano", "provenzal", "comino", "pimienta",
                     "pimenton", "laurel", "aji molido", "condiment"]):
        return "Almacén"

    # === 10) BEBIDAS ===
    if contiene(t, ["gaseos", "gaseo ", "cola ", " cola\t",
                     "sprite", "fanta", "pepsi", "mirinda", "7up", "7 up",
                     "manaos",
                     "agua min", "agua sin", "agua con", "agua s/gas",
                     "agua c/gas", "agua natur", "agua saborizada", "agua sab",
                     "isot", "powerade", "gatorade",
                     "jugo ", "nectar", "exprimido", "zuko", "clight",
                     "chocolatada", "leche choc", "cindor"]):
        return "Bebidas"

    # === 11) SNACKS Y GOLOSINAS ===
    if re.search(r"papas? fr", t):  # fix: papas fritas
        return "Snacks y golosinas"
    if contiene(t, ["alfajor", "chocolate", "choco ", "bombon", "caramelo",
                     "chupetin", "goma de ", "turron", "barra cereal",
                     "papas frit", "papa frita", "palit salad", "palitos",
                     "snack", "chizit", "nachos", "pororo", "pochoclo",
                     "mani ", "pasas uva",
                     "copa chocolate", "copa vainilla",
                     "oblea", "mana vainilla", "mana choc", "skarchit"]):
        return "Snacks y golosinas"

    return "Otros"


def paso_5_clasificar(canastables):
    """Aplica el clasificador consolidado."""
    print("\n🏷️  Clasificando productos en rubros...")

    canastables["rubro"] = canastables.apply(
        lambda r: clasificar_rubro(r["desc_norm"], r["marca"]), axis=1
    )

    distrib = canastables["rubro"].value_counts()
    total = len(canastables)
    print("\nDistribución por rubro:")
    for rubro, n in distrib.items():
        print(f"   {rubro:<22} {n:>5}  ({100*n/total:>4.1f}%)")

    return canastables


# =============================================================================
# PERSISTENCIA A DISCO (parquet)
# =============================================================================

def guardar_cache(dfs_dict):
    """Guarda dict de {nombre: df} a parquets en CACHE_DIR."""
    for nombre, df in dfs_dict.items():
        if df is None or df.empty:
            continue
        ruta = CACHE_DIR / f"{nombre}.parquet"
        # Convertir categorías a string para compatibilidad parquet
        df_save = df.copy()
        for col in df_save.select_dtypes(include=["category"]).columns:
            df_save[col] = df_save[col].astype(str)
        df_save.to_parquet(ruta, compression="snappy")
        print(f"   💾 Guardado: {ruta.name}  ({ruta.stat().st_size/(1024**2):.1f} MB)")


def cargar_cache(nombres):
    """Carga parquets desde CACHE_DIR. Devuelve dict {nombre: df} o None si falta alguno."""
    resultado = {}
    for n in nombres:
        ruta = CACHE_DIR / f"{n}.parquet"
        if not ruta.exists():
            return None
        resultado[n] = pd.read_parquet(ruta)
    return resultado


# =============================================================================
# PIPELINE PRINCIPAL
# =============================================================================

def pipeline():
    """
    Ejecuta todo el pipeline, con cache intermedio. Si el kernel crashea,
    se puede re-ejecutar y va a saltar los pasos ya completados.
    """

    # --- ETAPA A: carga base (la más pesada) ---
    cache_a = cargar_cache(["df_comercio", "df_sucursales", "df_precios"])
    if cache_a is not None:
        print("♻️  Cache ETAPA A encontrado, cargando...")
        df_comercio   = cache_a["df_comercio"]
        df_sucursales = cache_a["df_sucursales"]
        df_precios    = cache_a["df_precios"]
        # Re-convertir a category las columnas que habíamos convertido
        for col in ["comercio_bandera_nombre", "provincia", "sucursales_localidad",
                     "id_producto", "productos_descripcion", "productos_marca"]:
            if col in df_precios.columns:
                df_precios[col] = df_precios[col].astype("category")
    else:
        print("🚀 ETAPA A: carga desde cero")
        fecha_str, dia_dir, carpetas = paso_1_descomprimir()
        df_comercio, df_sucursales, df_productos = paso_2_cargar_y_optimizar(carpetas)
        df_comercio, df_sucursales, df_precios = paso_3_construir_df_precios(
            df_comercio, df_sucursales, df_productos
        )
        print("\n💾 Guardando cache ETAPA A...")
        guardar_cache({
            "df_comercio":   df_comercio,
            "df_sucursales": df_sucursales,
            "df_precios":    df_precios,
        })

    if DIAGNOSTICOS:
        print("\n📊 Memoria post ETAPA A:")
        reportar_memoria()

    # --- ETAPA B: canastables + clasificación ---
    cache_b = cargar_cache(["canastables"])
    if cache_b is not None:
        print("\n♻️  Cache ETAPA B encontrado, cargando...")
        canastables = cache_b["canastables"]
    else:
        print("\n🚀 ETAPA B: métricas y clasificación")
        canastables = paso_4_calcular_canastables(df_precios)
        canastables = paso_5_clasificar(canastables)
        print("\n💾 Guardando cache ETAPA B...")
        guardar_cache({"canastables": canastables})

    if DIAGNOSTICOS:
        print("\n📊 Memoria final:")
        reportar_memoria()

    print("\n✅ Pipeline completo. Variables disponibles:")
    print("   - df_comercio, df_sucursales, df_precios")
    print("   - canastables (con columna 'rubro')")

    return df_comercio, df_sucursales, df_precios, canastables


# =============================================================================
# EJECUCIÓN
# =============================================================================

if __name__ == "__main__" or True:  # se ejecuta también si pegás el bloque en Colab
    df_comercio, df_sucursales, df_precios, canastables = pipeline()

♻️  Cache ETAPA A encontrado, cargando...

📊 Memoria post ETAPA A:
   TOTAL                          0.0 MB  

♻️  Cache ETAPA B encontrado, cargando...

📊 Memoria final:
   TOTAL                          0.0 MB  

✅ Pipeline completo. Variables disponibles:
   - df_comercio, df_sucursales, df_precios
   - canastables (con columna 'rubro')


In [2]:
# =============================================================================
# EXPORTAR A GOOGLE DRIVE para usar desde otro notebook
# =============================================================================
from google.colab import drive
from pathlib import Path
import shutil

# Montamos Drive (te va a pedir autorización la primera vez)
drive.mount('/content/drive')

# Carpeta destino en Drive (se crea si no existe)
DRIVE_DIR = Path("/content/drive/MyDrive/sepa_analisis/2026-04-24")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Copiamos todos los parquets del cache local
CACHE_DIR = Path("/content/sepa_cache")
archivos_copiados = []
for parquet in CACHE_DIR.glob("*.parquet"):
    destino = DRIVE_DIR / parquet.name
    shutil.copy2(parquet, destino)
    peso_mb = destino.stat().st_size / (1024**2)
    archivos_copiados.append((parquet.name, peso_mb))
    print(f"   ✅ {parquet.name:<30} → {peso_mb:>7.2f} MB")

print(f"\n💾 Archivos exportados a: {DRIVE_DIR}")
print(f"   Total: {len(archivos_copiados)} archivos, "
      f"{sum(mb for _, mb in archivos_copiados):.1f} MB")

Mounted at /content/drive
   ✅ df_sucursales.parquet          →    0.19 MB
   ✅ df_precios.parquet             →  219.27 MB
   ✅ df_comercio.parquet            →    0.01 MB
   ✅ canastables.parquet            →    0.59 MB

💾 Archivos exportados a: /content/drive/MyDrive/sepa_analisis/2026-04-24
   Total: 4 archivos, 220.1 MB
